# Chest X-ray Classification — v2 (leak-free evaluation)

This is the corrected version of `chest_xray_classification.ipynb`. The models and ideas are the same; the **evaluation is fixed** so the test score reflects how the model would do on X-rays it has never seen.

| Problem in v1 | Fix in v2 |
|---|---|
| Augmented copies were created **before** the train/val/test split, so altered copies of one X-ray could land in both training and test | Split the **original** images first; augment the **training set only**. Validation and test contain only original, untouched images |
| Several X-rays come from the **same patient**, and a random split can put one patient on both sides | **Patient-level split** (`StratifiedGroupKFold`): every patient's images stay together in one set |
| PyTorch models used `random_split` on one dataset, so val/test images got the **training augmentations** | Separate folders and separate transforms: random augmentation for training, plain resize + normalise for val/test |
| Keras EfficientNet received images scaled to 0–1, but `tf.keras.applications.EfficientNet` already rescales internally and expects 0–255 | Feed raw 0–255 pixels to EfficientNet |
| Each model was evaluated a little differently | One shared evaluation for all four models: accuracy, macro F1, per-class report, confusion matrix, saved to `results/` |

**Run on Kaggle with a GPU** (Settings → Accelerator → GPU, and Internet **on** so pretrained weights can download). Add the dataset *Chest X-Ray Images (Pneumonia)* and set `INPUT_DIR` below if your path is different.

In [ ]:
import os, re, glob, json, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# ---------------- configuration ----------------
SMOKE = os.environ.get("SMOKE") == "1"          # tiny local test run; leave unset on Kaggle
INPUT_DIR = os.environ.get("INPUT_DIR", "/kaggle/input/chest-xray-pneumonia/chest_xray")
WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")
DATA_DIR = os.path.join(WORK_DIR, "dataset")      # train / val / test folders are created here
RESULTS_DIR = os.path.join(WORK_DIR, "results")
CLASSES = ["NORMAL", "PNEUMONIA_BACTERIAL", "PNEUMONIA_VIRAL"]
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 2 if SMOKE else 20
PATIENCE = 3
SEED = 42

random.seed(SEED); np.random.seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Input:", INPUT_DIR)

## 1. Collect the original images and label them (3 classes)
Works with the standard Kaggle layout (`train/`, `val/`, `test/` each holding `NORMAL/` and `PNEUMONIA/`) or a single merged `NORMAL/` + `PNEUMONIA/` folder — all original images are pooled and re-split by patient below. Pneumonia images are labelled bacterial or viral from their file names.

In [ ]:
def label_for(path):
    parent = os.path.basename(os.path.dirname(path)).upper()
    name = os.path.basename(path).lower()
    if parent == "NORMAL":
        return "NORMAL"
    if parent == "PNEUMONIA":
        if "bacteria" in name: return "PNEUMONIA_BACTERIAL"
        if "virus" in name:    return "PNEUMONIA_VIRAL"
    return None

def patient_id(path):
    """Group key so every image of one patient stays in the same split."""
    name = os.path.basename(path)
    m = re.match(r"(person\d+)_", name, re.I)          # person1_bacteria_1.jpeg
    if m: return m.group(1).lower()
    m = re.match(r"((?:NORMAL2-)?IM-\d+)", name, re.I)  # IM-0115-0001.jpeg, NORMAL2-IM-0537-0001.jpeg
    if m: return m.group(1).upper()
    return os.path.splitext(name)[0]

paths = [p for p in glob.glob(os.path.join(INPUT_DIR, "**", "*"), recursive=True)
         if p.lower().endswith((".jpeg", ".jpg", ".png")) and "__MACOSX" not in p]
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_for)
df = df.dropna(subset=["label"])
df["patient"] = df["path"].map(patient_id)
# some copies of the dataset contain a duplicated nested folder; keep one copy of each file
df["file"] = df["path"].map(os.path.basename)
df = df.drop_duplicates(subset=["label", "file"]).reset_index(drop=True)

print(f"{len(df)} images from {df['patient'].nunique()} patients")
print(df["label"].value_counts())

## 2. Patient-level, stratified 70 / 15 / 15 split — **before** any augmentation
`StratifiedGroupKFold` with 20 folds: 3 folds → test, 3 → validation, 14 → train. Class proportions stay close to the full dataset, and no patient appears in more than one set.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
fold = np.zeros(len(df), dtype=int)
for k, (_, idx) in enumerate(sgkf.split(df["path"], df["label"], groups=df["patient"])):
    fold[idx] = k
df["split"] = np.where(fold < 3, "test", np.where(fold < 6, "val", "train"))

# safety checks: no patient or file in two splits
assert df.groupby("patient")["split"].nunique().max() == 1, "patient leak between splits"
assert not df["path"].duplicated().any()

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
for _, r in df.iterrows():
    dest = os.path.join(DATA_DIR, r["split"], r["label"])
    os.makedirs(dest, exist_ok=True)
    shutil.copy(r["path"], os.path.join(dest, os.path.basename(r["path"])))

split_counts = pd.crosstab(df["split"], df["label"]).loc[["train", "val", "test"]]
print(split_counts)

## 3. Balance the **training set only** with Albumentations
Minority classes in `train/` are topped up with augmented copies until they match the largest training class. Validation and test are left untouched.

In [ ]:
import cv2
import albumentations as A

try:    # Albumentations 2.x
    crop = A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=0.5)
except Exception:  # Albumentations 1.x
    crop = A.RandomResizedCrop(height=IMG_SIZE, width=IMG_SIZE, scale=(0.8, 1.0), p=0.5)

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(p=0.2),
    crop,
])

train_dir = os.path.join(DATA_DIR, "train")
target = max(len(os.listdir(os.path.join(train_dir, c))) for c in CLASSES)
rng = np.random.default_rng(SEED)
for c in CLASSES:
    folder = os.path.join(train_dir, c)
    originals = sorted(os.listdir(folder))
    for i in range(target - len(originals)):
        src = rng.choice(originals)
        img = cv2.cvtColor(cv2.imread(os.path.join(folder, src)), cv2.COLOR_BGR2RGB)
        out = augment(image=img)["image"]
        cv2.imwrite(os.path.join(folder, f"aug_{i}_{src}"), cv2.cvtColor(out, cv2.COLOR_RGB2BGR))

counts = {s: {c: len(os.listdir(os.path.join(DATA_DIR, s, c))) for c in CLASSES} for s in ["train", "val", "test"]}
print(pd.DataFrame(counts).T)
# augmented files exist only in train
assert not any(f.startswith("aug_") for s in ["val", "test"] for c in CLASSES for f in os.listdir(os.path.join(DATA_DIR, s, c)))

In [ ]:
ax = pd.DataFrame(counts).T.plot.bar(figsize=(8, 4), rot=0, color=["#4C78A8", "#E45756", "#F58518"])
ax.set_title("Images per class and split (train balanced, val/test original only)")
ax.set_ylabel("Images")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "split_counts.png"), dpi=120); plt.show()

## 4. Shared evaluation helper

In [ ]:
RESULTS = []

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n=== {name}: test accuracy {acc:.4f} | macro F1 {macro_f1:.4f} ===")
    rep = classification_report(y_true, y_pred, labels=range(len(CLASSES)), target_names=CLASSES, digits=3, output_dict=True)
    print(classification_report(y_true, y_pred, labels=range(len(CLASSES)), target_names=CLASSES, digits=3))
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"{name} — confusion matrix")
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"cm_{name}.png"), dpi=120); plt.show()
    row = {"model": name, "test_accuracy": round(acc, 4), "macro_f1": round(macro_f1, 4)}
    row.update({f"f1_{c}": round(rep[c]["f1-score"], 4) for c in CLASSES})
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(os.path.join(RESULTS_DIR, "results_v2.csv"), index=False)

def plot_curves(name, hist):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["train_acc"], label="train"); ax[0].plot(hist["val_acc"], label="val"); ax[0].set_title(f"{name} accuracy")
    ax[1].plot(hist["train_loss"], label="train"); ax[1].plot(hist["val_loss"], label="val"); ax[1].set_title(f"{name} loss")
    for a in ax: a.set_xlabel("epoch"); a.legend()
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"curves_{name}.png"), dpi=120); plt.show()

## 5. EfficientNet-B5 (TensorFlow / Keras)
Raw 0–255 pixels (the model rescales internally), augmentation layers active only during training, early stopping on validation loss.

In [ ]:
try:
    import tensorflow as tf
    HAVE_TF = True
except ImportError:
    HAVE_TF = False
    print("TensorFlow not installed - skipping EfficientNet")

if HAVE_TF:
    tf.keras.utils.set_random_seed(SEED)
    load = lambda split, shuffle: tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA_DIR, split), labels="inferred", label_mode="categorical", class_names=CLASSES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED)
    tf_train, tf_val, tf_test = load("train", True), load("val", False), load("test", False)
    AUTOTUNE = tf.data.AUTOTUNE
    tf_train, tf_val, tf_test = [d.prefetch(AUTOTUNE) for d in (tf_train, tf_val, tf_test)]

    aug_layers = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(15 / 360),
        tf.keras.layers.RandomTranslation(0.1, 0.1),
        tf.keras.layers.RandomZoom(0.1),
    ])
    base = tf.keras.applications.EfficientNetB5(weights=None if SMOKE else "imagenet",
                                                include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = aug_layers(inputs)                       # only active in training
    x = base(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(512, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    effnet = tf.keras.Model(inputs, outputs)
    effnet.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])

    h = effnet.fit(tf_train, validation_data=tf_val, epochs=EPOCHS, callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6)])

    y_true = np.concatenate([np.argmax(y, 1) for _, y in tf_test])
    y_pred = np.argmax(effnet.predict(tf_test), 1)
    plot_curves("EfficientNetB5", {"train_acc": h.history["accuracy"], "val_acc": h.history["val_accuracy"],
                                   "train_loss": h.history["loss"], "val_loss": h.history["val_loss"]})
    report("EfficientNetB5", y_true, y_pred)

## 6. ViT-B/16, ResNet-50 and Xception (PyTorch)
Same folders for every model; training transforms only on `train/`.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import timm

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = T.Compose([T.Resize((256, 256)), T.RandomCrop(IMG_SIZE), T.RandomHorizontalFlip(),
                      T.RandomRotation(15), T.ColorJitter(brightness=0.2, contrast=0.2), T.ToTensor(), norm])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), norm])

ds = {s: ImageFolder(os.path.join(DATA_DIR, s), transform=train_tf if s == "train" else eval_tf) for s in ["train", "val", "test"]}
assert ds["train"].classes == CLASSES, ds["train"].classes
workers = 0 if SMOKE else 2
dl = {s: DataLoader(ds[s], batch_size=BATCH_SIZE, shuffle=(s == "train"), num_workers=workers, pin_memory=True) for s in ds}

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total, correct, loss_sum, preds, labels = 0, 0, 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            p = out.argmax(1)
            loss_sum += loss.item() * y.size(0); total += y.size(0); correct += (p == y).sum().item()
            preds.append(p.cpu()); labels.append(y.cpu())
    return loss_sum / total, correct / total, torch.cat(labels).numpy(), torch.cat(preds).numpy()

def train_torch(name, model, lr, weight_decay, scheduler_fn):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = scheduler_fn(optimizer)
    hist = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best, wait, ckpt = -1, 0, os.path.join(WORK_DIR, f"best_{name}.pth")
    for epoch in range(EPOCHS):
        tl, ta, _, _ = run_epoch(model, dl["train"], criterion, optimizer)
        vl, va, _, _ = run_epoch(model, dl["val"], criterion)
        for k, v in zip(hist, (ta, va, tl, vl)): hist[k].append(v)
        print(f"{name} epoch {epoch+1}: train acc {ta:.4f} loss {tl:.4f} | val acc {va:.4f} loss {vl:.4f}")
        if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau): scheduler.step(va)
        else: scheduler.step()
        if va > best:
            best, wait = va, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("early stopping"); break
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    _, _, y_true, y_pred = run_epoch(model, dl["test"], criterion)
    plot_curves(name, hist)
    report(name, y_true, y_pred)
    return model

In [ ]:
def make_vit():
    m = models.vit_b_16(weights=None if SMOKE else models.ViT_B_16_Weights.DEFAULT)
    m.heads.head = nn.Linear(m.heads.head.in_features, len(CLASSES))
    return m

def make_resnet50():
    m = models.resnet50(weights=None if SMOKE else models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    return m

def make_xception():
    return timm.create_model("legacy_xception", pretrained=not SMOKE, num_classes=len(CLASSES))

cosine = lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
plateau = lambda opt: optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)

TORCH_RUNS = [
    ("ViT_B16",  make_vit,      5e-5, 5e-3, cosine),
    ("ResNet50", make_resnet50, 1e-5, 1e-3, plateau),
    ("Xception", make_xception, 5e-5, 5e-3, cosine),
]
if SMOKE:   # small CPU-only check of the training loop
    TORCH_RUNS = [("ResNet18_smoke", lambda: nn.Sequential(models.resnet18(weights=None), nn.Linear(1000, len(CLASSES))), 1e-3, 0.0, cosine)]

for name, make, lr, wd, sched in TORCH_RUNS:
    train_torch(name, make(), lr, wd, sched)
    if device.type == "cuda": torch.cuda.empty_cache()

## 7. Compare all models

In [ ]:
results = pd.DataFrame(RESULTS).sort_values("test_accuracy", ascending=False)
display(results) if "display" in globals() else print(results)
results.to_csv(os.path.join(RESULTS_DIR, "results_v2.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "split_counts.json"), "w") as f:
    json.dump(counts, f, indent=1)

ax = results.plot.bar(x="model", y=["test_accuracy", "macro_f1"], figsize=(8, 4), rot=0, color=["#4C78A8", "#72B7B2"])
ax.set_ylim(0, 1); ax.set_title("v2 test results (patient-level split, original images only)")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "model_comparison.png"), dpi=120); plt.show()

## Notes
- Expect v2 scores to be **lower** than v1 — that is the point: v1 was partly testing on near-copies of training images. v2 numbers are the honest ones to quote.
- Download the `results/` folder from Kaggle's output panel and add `results_v2.csv` and the PNGs to the repo's `images/` folder.
- This is an educational project, not a diagnostic tool.